# Megazeka-Kesir v2.0 · Qwen3.5-4B egitimi (Colab L4)

v1.0 (v1.3) verisi ve ayarlariyla ayni egitim, yalniz taban degisir: Qwen2.5-1.5B -> **Qwen3.5-4B** (Apache-2.0).
Calisma zamani turu olarak **L4** secin. T4'te bfloat16 yok; Qwen3.5 float32'ye duser ve egitim ~4 kat yavaslar.

Sira: **1-4 hazirlik + duman testi** (birkac dakika) -> **5 kisa cikarim** -> **6 tam egitim** (L4'te ~3 saat).
Kopma olursa 1-3'u tekrar calistir, sonra 6'da `DEVAM = True` yapip calistir (son checkpoint'ten surer).
Ciktilar Drive'da: `MyDrive/megazeka-kesir-colab/` (models/, reports/).
Bu not defteri `colab/notebook_olustur.py` ile yerel dosyalardan uretildi; elle degil orada duzenle.

In [ ]:
# 1) GPU + Drive
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
from google.colab import drive
drive.mount('/content/drive')
KOK = '/content/drive/MyDrive/megazeka-kesir-colab'
for d in ('training', 'configs', 'data/v13', 'models', 'reports'):
    os.makedirs(f'{KOK}/{d}', exist_ok=True)
os.environ['UNSLOTH_COMPILE_LOCATION'] = '/content/unsloth_compiled_cache'  # Drive'a degil, yerel diske
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUTF8'] = '1'
os.chdir(KOK)
print('KOK =', KOK)

In [ ]:
# 2) Kurulum (Unsloth + Qwen3.5 dogrusal dikkat cekirdekleri)
!pip install -q --upgrade unsloth
!pip install -q flash-linear-attention
import importlib.metadata as im
for p in ('unsloth', 'unsloth_zoo', 'trl', 'transformers', 'peft', 'bitsandbytes', 'torch', 'flash-linear-attention'):
    try: print(f'{p:24s}', im.version(p))
    except Exception as e: print(f'{p:24s} YOK ({e})')

In [ ]:
# 3) Egitim betigi, ayarlar ve veri (HF pandakingpunc/megazeka-kesir-tr-steps@v1.0, sha256 denetimli)
import hashlib, json, shutil
from huggingface_hub import hf_hub_download

TRAIN_PY = '"""Megazeka-Kesir QLoRA egitimi (Unsloth + TRL SFTTrainer).\n\nKullanim (Unsloth Studio ortaminin Python\'u ile, proje kokunden):\n  python -X utf8 training/train.py --config configs/v1.json --ad v1 [--max-steps N] [--max-val-examples N] [--resume]\n\nStudio\'nun v1\'de kullandigi yolu izler: FastLanguageModel (4-bit) + get_peft_model, mesajlar\ntokenizer sohbet sablonuyla "text" alanina duzlestirilir, SFTTrainer olusturulduktan sonra\nunsloth.chat_templates.train_on_responses_only ile yalniz asistan yanitlarinda kayip hesaplanir.\n\nCiktilar:\n  models/checkpoints/<ad>/checkpoint-N   kontrol noktalari\n  models/<ad>/                           son adaptor + tokenizer + egitim_ayari.json\n  models/<ad>-best/                      en dusuk val kaybindaki adaptor (best_info.json ile)\n  reports/gecmis-<ad>.jsonl              baslangic satiri + her log adiminda kayip gecmisi\n  Test degerlendirmesi ayri bir komutla calistirilir.\n"""\nimport os\nimport sys\n\nKOK_ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\n# Unsloth\'un derlenmis trainer onbellegi cwd\'ye degil proje icindeki .cache\'e yazilsin.\nos.environ.setdefault("UNSLOTH_COMPILE_LOCATION", os.path.join(KOK_, ".cache", "unsloth_compiled_cache"))\nos.environ.setdefault("PYTHONUTF8", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport argparse\nimport datetime\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nKOK = Path(KOK_)\nVARSAYILAN = {\n    "logging_steps": 10,\n    "optim": "adamw_8bit",\n    "gradient_checkpointing": "unsloth",\n    "save_total_limit": 3,\n}\nINSTRUCTION_PART = "<|im_start|>user\\n"\nRESPONSE_PART = "<|im_start|>assistant\\n"\nCEVAP_ONEKLERI = ("İşlem:", "Soru:", "Kesir:", "Sonuç:", "Okunuş:")\n\n\ndef yol(p):\n    p = Path(p)\n    return p if p.is_absolute() else KOK / p\n\n\ndef satir_say(p):\n    with open(p, encoding="utf-8") as f:\n        return sum(1 for s in f if s.strip())\n\n\ndef adim_hesapla(n, batch, ga, epochs, max_steps):\n    if max_steps and max_steps > 0:\n        return int(max_steps)\n    return math.ceil(math.ceil(n / batch) / ga) * epochs\n\n\ndef adim_korumasi(n, cfg, max_steps, etiket):\n    toplam = adim_hesapla(n, cfg["batch_size"], cfg["gradient_accumulation"], cfg["epochs"], max_steps)\n    if toplam < cfg["warmup_steps"]:\n        sys.exit(\n            f"HATA ({etiket}): toplam optimizer adimi ({toplam}) < warmup_steps ({cfg[\'warmup_steps\']}). "\n            f"Ornek={n}, batch={cfg[\'batch_size\']}, birikim={cfg[\'gradient_accumulation\']}, epoch={cfg[\'epochs\']}, "\n            f"max_steps={max_steps}. Egitim baslatilmadi."\n        )\n    return toplam\n\n\ndef son_checkpoint(cikti):\n    if not cikti.is_dir():\n        return None\n    cps = [d for d in cikti.glob("checkpoint-*") if d.name.split("-")[-1].isdigit()]\n    return max(cps, key=lambda d: int(d.name.split("-")[-1])) if cps else None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", required=True)\n    ap.add_argument("--ad", required=True)\n    ap.add_argument("--max-steps", type=int, default=None)\n    ap.add_argument("--resume", action="store_true")\n    ap.add_argument("--max-val-examples", type=int)\n    args = ap.parse_args()\n\n    with open(yol(args.config), encoding="utf-8") as f:\n        cfg_dosya = json.load(f)\n    cfg = {**VARSAYILAN, **cfg_dosya}\n    if args.max_steps is not None and args.max_steps < 1:\n        ap.error("--max-steps pozitif olmali.")\n    if args.max_val_examples is not None and args.max_val_examples < 1:\n        ap.error("--max-val-examples pozitif olmali.")\n    if args.max_steps is not None:\n        cfg["max_steps"] = args.max_steps\n    max_steps = cfg.get("max_steps", 0) or 0\n    ad = args.ad\n    if (max_steps > 0 or args.max_val_examples is not None) and not ad.startswith("pilot-"):\n        ap.error("Sinirli calistirmalarda --ad pilot- ile baslamali.")\n\n    train_yol, val_yol = yol(cfg["train"]), yol(cfg["val"])\n    for p in (train_yol, val_yol):\n        if not p.is_file():\n            sys.exit(f"HATA: veri dosyasi yok: {p}")\n\n    cikti = KOK / "models" / "checkpoints" / ad\n    son_adaptor = KOK / "models" / ad\n    en_iyi_dir = KOK / "models" / f"{ad}-best"\n    gecmis_yol = KOK / "reports" / f"gecmis-{ad}.jsonl"\n    gecmis_yol.parent.mkdir(parents=True, exist_ok=True)\n\n    # Ucuz erken koruma: ham satir sayisiyla (model yuklemeden once).\n    n_ham = satir_say(train_yol)\n    if not n_ham or not satir_say(val_yol):\n        sys.exit("HATA: egitim ve val veri dosyalari bos olamaz.")\n    adim_korumasi(n_ham, cfg, max_steps, "ham veri")\n\n    resume_cp = None\n    if args.resume:\n        resume_cp = son_checkpoint(cikti)\n        if resume_cp is None:\n            sys.exit(f"HATA: --resume verildi ama {cikti} altinda checkpoint yok.")\n\n    # ---- agir importlar (unsloth, trl\'den ve transformers\'tan once) ----\n    import unsloth  # noqa: F401\n    from unsloth import FastLanguageModel\n    from unsloth.chat_templates import train_on_responses_only\n    import torch\n    import transformers\n    import trl\n    from datasets import load_dataset\n    from transformers import TrainerCallback\n    from trl import SFTConfig, SFTTrainer\n\n    print(f"[surum] unsloth={unsloth.__version__} trl={trl.__version__} transformers={transformers.__version__} torch={torch.__version__}")\n    trl_beklenen = cfg.get("trl_surumu", "0.23.1")\n    if trl_beklenen and trl.__version__ != trl_beklenen:\n        sys.exit(f"HATA: TRL {trl_beklenen} gerekli. Unsloth Studio ortamini kullanin.")\n    if not torch.cuda.is_available():\n        sys.exit("HATA: CUDA yok.")\n\n    # ---- model ----\n    # Gorsel-metin tabanlar (or. Qwen3.5) icin "yukleyici": "FastModel"; yalniz metin egitilir.\n    if cfg.get("yukleyici", "FastLanguageModel") == "FastModel":\n        from unsloth import FastModel as Yukleyici\n    else:\n        Yukleyici = FastLanguageModel\n    model, tokenizer = Yukleyici.from_pretrained(\n        model_name=cfg["taban"],\n        max_seq_length=cfg["max_seq_length"],\n        dtype=None,\n        load_in_4bit=True,\n        use_gradient_checkpointing=cfg["gradient_checkpointing"],\n    )\n    if hasattr(tokenizer, "tokenizer"):  # isleyici (processor) donduyse metin tokenizer\'i kullan\n        print(f"[taban] isleyici {type(tokenizer).__name__} -> {type(tokenizer.tokenizer).__name__}")\n        tokenizer = tokenizer.tokenizer\n    model = Yukleyici.get_peft_model(\n        model,\n        r=cfg["rank"],\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n        lora_alpha=cfg["alpha"],\n        lora_dropout=cfg["dropout"],\n        bias="none",\n        use_gradient_checkpointing=cfg["gradient_checkpointing"],\n        random_state=cfg["seed"],\n        use_rslora=False,\n    )\n\n    # ---- veri: mesajlar -> sohbet sablonuyla "text" ----\n    ds = load_dataset("json", data_files={"train": str(train_yol), "validation": str(val_yol)})\n    if len(ds["train"]) != n_ham:\n        sys.exit(f"HATA: egitim JSONL\'de {n_ham} satir, yuklenen veri {len(ds[\'train\'])} ornek.")\n    if args.max_val_examples is not None:\n        ds["validation"] = ds["validation"].select(range(min(args.max_val_examples, len(ds["validation"]))))\n\n    beklenen = {}\n    for etiket, dataset in ds.items():\n        def metne(ornek, i):\n            mesajlar = ornek["messages"]\n            if [m["role"] for m in mesajlar] != ["user", "assistant"]:\n                raise ValueError(f"{etiket}#{i + 1}: user ve assistant mesajlari sirayla gerekli.")\n            for mesaj in mesajlar:\n                icerik = mesaj["content"]\n                if not isinstance(icerik, str) or not icerik.strip():\n                    raise ValueError(f"{etiket}#{i + 1}: bos veya gecersiz mesaj.")\n                if "<|im_start|>" in icerik or "<|im_end|>" in icerik:\n                    raise ValueError(f"{etiket}#{i + 1}: mesaj iceriginde ChatML rol isareti.")\n            return {"text": tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=False)}\n\n        metinler = dataset.map(metne, with_indices=True, remove_columns=dataset.column_names, num_proc=None)\n        ids = []\n        cevaplar = []\n        en_uzun = 0\n        for bas in range(0, len(dataset), 256):\n            metin_grubu = metinler[bas:bas + 256]["text"]\n            token_grubu = tokenizer(metin_grubu, add_special_tokens=False, truncation=False)["input_ids"]\n            for i, tokenlar in enumerate(token_grubu, start=bas):\n                uzunluk = len(tokenlar)\n                if uzunluk > cfg["max_seq_length"]:\n                    raise ValueError(\n                        f"{etiket}#{i + 1}: {uzunluk} token, baglam siniri {cfg[\'max_seq_length\']}."\n                    )\n                en_uzun = max(en_uzun, uzunluk)\n                ids.append(tokenlar)\n                cevaplar.append(dataset[i]["messages"][1]["content"])\n        beklenen[etiket] = (ids, cevaplar)\n        ds[etiket] = metinler\n        print(f"[veri:{etiket}] ornek={len(dataset)} en_uzun={en_uzun} token")\n    train_ds, val_ds = ds["train"], ds["validation"]\n\n    bf16 = torch.cuda.is_bf16_supported()\n    sft_args = dict(\n        per_device_train_batch_size=cfg["batch_size"],\n        per_device_eval_batch_size=cfg["batch_size"],\n        gradient_accumulation_steps=cfg["gradient_accumulation"],\n        learning_rate=cfg["learning_rate"],\n        lr_scheduler_type=cfg["scheduler"],\n        warmup_steps=cfg["warmup_steps"],\n        weight_decay=cfg["weight_decay"],\n        optim=cfg["optim"],\n        bf16=bf16,\n        fp16=not bf16,\n        seed=cfg["seed"],\n        logging_steps=cfg["logging_steps"],\n        eval_strategy="steps",\n        eval_steps=cfg["eval_steps"],\n        save_strategy="steps",\n        save_steps=cfg["save_steps"],\n        save_total_limit=cfg["save_total_limit"],\n        output_dir=str(cikti),\n        report_to="none",\n        dataset_text_field="text",\n        max_seq_length=cfg["max_seq_length"],\n        max_length=cfg["max_seq_length"],  # TRL 0.23 varsayilani 1024; Unsloth once bunu okur\n        packing=False,\n        dataset_num_proc=1,        # Windows: multiprocessing sorun cikarir\n        dataloader_num_workers=0,  # Windows\n        gradient_checkpointing=True,\n    )\n    if max_steps > 0:\n        sft_args["max_steps"] = max_steps\n    else:\n        sft_args["num_train_epochs"] = cfg["epochs"]\n\n    # ---- geri cagirimlar: kayip gecmisi + en iyi val adaptoru ----\n    gecmis_son = {"loss": None, "eval_loss": None}\n\n    class Gecmis(TrainerCallback):\n        def on_log(self, args_, state, control, logs=None, **kw):\n            if not logs or not ("loss" in logs or "eval_loss" in logs):\n                return\n            kayit = {\n                "adim": state.global_step,\n                "epoch": logs.get("epoch"),\n                "loss": logs.get("loss"),\n                "eval_loss": logs.get("eval_loss"),\n                "lr": logs.get("learning_rate"),\n                "grad_norm": logs.get("grad_norm"),\n            }\n            if kayit["loss"] is not None:\n                gecmis_son["loss"] = kayit["loss"]\n            if kayit["eval_loss"] is not None:\n                gecmis_son["eval_loss"] = kayit["eval_loss"]\n            with open(gecmis_yol, "a", encoding="utf-8") as f:\n                f.write(json.dumps(kayit, ensure_ascii=False) + "\\n")\n\n    en_iyi = {"loss": float("inf"), "adim": None}\n    if args.resume and (en_iyi_dir / "best_info.json").is_file():\n        bilgi = json.loads((en_iyi_dir / "best_info.json").read_text(encoding="utf-8"))\n        en_iyi = {"loss": bilgi["eval_loss"], "adim": bilgi["adim"]}\n\n    class EnIyi(TrainerCallback):\n        trainer = None\n\n        def on_evaluate(self, args_, state, control, metrics=None, **kw):\n            el = (metrics or {}).get("eval_loss")\n            if el is None or not math.isfinite(el) or el >= en_iyi["loss"]:\n                return\n            en_iyi.update(loss=el, adim=state.global_step)\n            en_iyi_dir.mkdir(parents=True, exist_ok=True)\n            self.trainer.model.save_pretrained(str(en_iyi_dir))\n            tokenizer.save_pretrained(str(en_iyi_dir))\n            (en_iyi_dir / "best_info.json").write_text(\n                json.dumps({"adim": state.global_step, "eval_loss": el}, ensure_ascii=False, indent=2), encoding="utf-8")\n            print(f"[en iyi] adim {state.global_step}: eval_loss={el:.4f} -> {en_iyi_dir}")\n\n    gecmis_cb, en_iyi_cb = Gecmis(), EnIyi()\n    trainer = SFTTrainer(\n        model=model,\n        tokenizer=tokenizer,\n        train_dataset=train_ds,\n        eval_dataset=val_ds,\n        args=SFTConfig(**sft_args),\n        callbacks=[gecmis_cb, en_iyi_cb],\n    )\n    en_iyi_cb.trainer = trainer\n\n    # ---- yalniz asistan yanitlarinda kayip (Studio ile ayni yol) ----\n    trainer = train_on_responses_only(\n        trainer,\n        instruction_part=cfg.get("instruction_part", INSTRUCTION_PART),\n        response_part=cfg.get("response_part", RESPONSE_PART),\n        num_proc=1,\n    )\n    n_train, n_val = len(trainer.train_dataset), len(trainer.eval_dataset)\n    for etiket, dataset in (("train", trainer.train_dataset), ("validation", trainer.eval_dataset)):\n        ids_beklenen, cevaplar = beklenen[etiket]\n        if len(dataset) != len(ids_beklenen):\n            raise ValueError(f"{etiket}: {len(ids_beklenen)} ornekten {len(dataset)} ornek kaldi.")\n        onek_hatasi = 0\n        for i, satir in enumerate(dataset):\n            ids, labels = satir["input_ids"], satir["labels"]\n            if ids != ids_beklenen[i] or len(labels) != len(ids):\n                raise ValueError(f"{etiket}#{i + 1}: tokenlar degisti, kesildi veya etiket uzunlugu uyusmuyor.")\n            egitilen = [j for j, label in enumerate(labels) if label != -100]\n            if not egitilen or egitilen[0] == 0 or egitilen != list(range(egitilen[0], egitilen[-1] + 1)):\n                raise ValueError(f"{etiket}#{i + 1}: yanit maskesi bos veya kesintili.")\n            if any(labels[j] != ids[j] for j in egitilen):\n                raise ValueError(f"{etiket}#{i + 1}: egitim etiketleri tokenlarla uyusmuyor.")\n            metin = tokenizer.decode([labels[j] for j in egitilen], clean_up_tokenization_spaces=False)\n            if metin != cevaplar[i] + "<|im_end|>":\n                raise ValueError(f"{etiket}#{i + 1}: maske tam asistan cevabini ve bitis tokenini kapsamiyor.")\n            onek_hatasi += not cevaplar[i].startswith(CEVAP_ONEKLERI)\n        print(f"[maske:{etiket}] {len(dataset)} ornek dogrulandi, beklenen onek disinda={onek_hatasi}")\n    del beklenen\n\n    # ---- adim bilgisi + koruma (maskeleme sonrasi gercek ornek sayisiyla) ----\n    efektif = cfg["batch_size"] * cfg["gradient_accumulation"]\n    toplam_adim = adim_korumasi(n_train, cfg, max_steps, "maske sonrasi")\n    bilgi = {\n        "tip": "baslangic",\n        "tarih": datetime.datetime.now().isoformat(timespec="seconds"),\n        "ad": ad,\n        "ornek_sayisi_egitim": n_train,\n        "ornek_sayisi_val": n_val,\n        "batch_size": cfg["batch_size"],\n        "gradient_accumulation": cfg["gradient_accumulation"],\n        "efektif_batch": efektif,\n        "epoch": cfg["epochs"],\n        "max_steps": max_steps or None,\n        "toplam_optimizer_adimi": toplam_adim,\n        "warmup_steps": cfg["warmup_steps"],\n        "surumler": {"unsloth": unsloth.__version__, "trl": trl.__version__, "transformers": transformers.__version__},\n    }\n    print(f"[plan] egitim={n_train} val={n_val} efektif_batch={efektif} toplam_optimizer_adimi={toplam_adim} warmup={cfg[\'warmup_steps\']}")\n    mod = "a" if args.resume else "w"\n    with open(gecmis_yol, mod, encoding="utf-8") as f:\n        if args.resume:\n            bilgi["tip"] = "devam"\n            bilgi["checkpoint"] = resume_cp.name\n        f.write(json.dumps(bilgi, ensure_ascii=False) + "\\n")\n\n    # ---- egitim ----\n    torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    sonuc = trainer.train(resume_from_checkpoint=str(resume_cp) if resume_cp else None)\n    sure = time.time() - t0\n    if trainer.state.global_step != toplam_adim:\n        raise RuntimeError(f"Tamamlanan adim {trainer.state.global_step} != planlanan {toplam_adim}.")\n    # son adimda degerlendirme yapilmadiysa yap (best callback\'i de tetikler)\n    if not any(h.get("eval_loss") is not None and h["step"] == trainer.state.global_step for h in trainer.state.log_history):\n        trainer.evaluate()\n    if not all(x is not None and math.isfinite(x) for x in gecmis_son.values()):\n        raise RuntimeError(f"Egitim veya val kaybi gecersiz: {gecmis_son}")\n    vram = torch.cuda.max_memory_allocated() / 2**20\n    vram_ayrilan = torch.cuda.max_memory_reserved() / 2**20\n    adim_sn = trainer.state.global_step / sure if sure else None\n    print(f"[bitti] adim={trainer.state.global_step} sure={sure:.1f}s adim/sn={adim_sn:.3f} "\n          f"son_loss={gecmis_son[\'loss\']} son_eval_loss={gecmis_son[\'eval_loss\']}")\n    print(f"[vram] tepe max_memory_allocated={vram:.0f} MiB (reserved {vram_ayrilan:.0f} MiB)")\n\n    # ---- son adaptor ----\n    son_adaptor.mkdir(parents=True, exist_ok=True)\n    trainer.model.save_pretrained(str(son_adaptor))\n    tokenizer.save_pretrained(str(son_adaptor))\n    ayar = {\n        "ad": ad,\n        "config": cfg,\n        "max_val_examples": args.max_val_examples,\n        "tarih": datetime.datetime.now().isoformat(timespec="seconds"),\n        "toplam_adim": trainer.state.global_step,\n        "son_egitim_kaybi": gecmis_son["loss"],\n        "son_degerlendirme_kaybi": gecmis_son["eval_loss"],\n        "en_iyi_degerlendirme_kaybi": None if en_iyi["adim"] is None else en_iyi["loss"],\n        "en_iyi_adim": en_iyi["adim"],\n        "egitim_ornek_sayisi": n_train,\n        "val_ornek_sayisi": n_val,\n        "efektif_batch": efektif,\n        "egitim_suresi_sn": round(sure, 1),\n        "adim_per_sn": None if adim_sn is None else round(adim_sn, 4),\n        "vram_tepe_mib": round(vram),\n        "surumler": bilgi["surumler"],\n    }\n    (son_adaptor / "egitim_ayari.json").write_text(json.dumps(ayar, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(f"[kayit] son adaptor -> {son_adaptor}" + (f"; en iyi (adim {en_iyi[\'adim\']}, eval_loss {en_iyi[\'loss\']:.4f}) -> {en_iyi_dir}" if en_iyi["adim"] else ""))\n\n\nif __name__ == "__main__":\n    main()\n'
AYAR = {'taban': 'unsloth/Qwen3.5-4B', 'train': 'data/v13/train.jsonl', 'val': 'data/v13/val.jsonl', 'max_seq_length': 1792, 'epochs': 2, 'batch_size': 8, 'gradient_accumulation': 2, 'learning_rate': 0.0002, 'warmup_steps': 50, 'scheduler': 'linear', 'weight_decay': 0.001, 'rank': 16, 'alpha': 32, 'dropout': 0.0, 'eval_steps': 250, 'save_steps': 125, 'seed': 42, 'trl_surumu': None, 'response_part': '<|im_start|>assistant\n<think>\n\n</think>\n\n', 'not': "v1.3 verisi ve ayarlari (8 x 2 dahil); yalniz taban degisir. Colab L4'te bf16 ile egitildi (T4'te bf16 yok, Qwen3.5 fp32'ye dusup ~20 sn/adim). 125 adimda bir kayit. Cikarimda enable_thinking=False."}
PILOT = {'taban': 'unsloth/Qwen3.5-4B', 'train': 'data/v13/train.jsonl', 'val': 'data/v13/val.jsonl', 'max_seq_length': 1792, 'epochs': 2, 'batch_size': 8, 'gradient_accumulation': 2, 'learning_rate': 0.0002, 'warmup_steps': 5, 'scheduler': 'linear', 'weight_decay': 0.001, 'rank': 16, 'alpha': 32, 'dropout': 0.0, 'eval_steps': 10, 'save_steps': 10, 'seed': 42, 'trl_surumu': None, 'response_part': '<|im_start|>assistant\n<think>\n\n</think>\n\n', 'not': 'Duman testi: 20 adim, 16 val ornegi.'}
HASH = {'train.jsonl': '6762d28c8628a6111b2fca347bb5df69b36d63aa67059ea8a9291248e141da24', 'validation.jsonl': '49345d5fa3cf6b8fee720cefd11d46b0f966d402e72bc3bb6107ab8e67c9a810', 'test.jsonl': '13754aa34189a2359778dd5d44365be57c54e1f5f379a2c6990638536aa631ab'}
YEREL = {'train.jsonl': 'train.jsonl', 'validation.jsonl': 'val.jsonl', 'test.jsonl': 'test.jsonl'}

open(f'{KOK}/training/train.py', 'w', encoding='utf-8').write(TRAIN_PY)
json.dump(AYAR, open(f'{KOK}/configs/v2-q35-4b.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
json.dump(PILOT, open(f'{KOK}/configs/pilot-q35-4b.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
for hf_ad, yerel in YEREL.items():
    hedef = f'{KOK}/data/v13/{yerel}'
    if not os.path.isfile(hedef) or hashlib.sha256(open(hedef, 'rb').read()).hexdigest() != HASH[hf_ad]:
        shutil.copy(hf_hub_download('pandakingpunc/megazeka-kesir-tr-steps', hf_ad, repo_type='dataset', revision='v1.0'), hedef)
    h = hashlib.sha256(open(hedef, 'rb').read()).hexdigest()
    assert h == HASH[hf_ad], f'{yerel}: sha256 uyusmuyor {h}'
    print(f'{yerel:12s} sha256 OK  {sum(1 for _ in open(hedef, encoding="utf-8"))} satir')

In [ ]:
# 4) DUMAN TESTI: 20 adim, 16 val ornegi -> maske denetimi + egitim + kayit
!rm -f reports/gecmis-pilot-q35-4b.jsonl
!python -X utf8 training/train.py --config configs/pilot-q35-4b.json --ad pilot-q35-4b --max-steps 20 --max-val-examples 16 2>&1 | tee reports/pilot-q35-4b.log | grep -v -E "^\s*$|it/s\]$"
import json
a = json.load(open('models/pilot-q35-4b/egitim_ayari.json'))
sn_adim = 1 / a['adim_per_sn']
print(f"\nVRAM tepe {a['vram_tepe_mib']} MiB · {sn_adim:.1f} sn/adim · tam egitim (2250 adim, eval haric) ~ {2250*sn_adim/3600:.1f} saat")

In [ ]:
# 5) Kisa cikarim: pilot adaptoru yukle, Turkce kesir sorusu sor (20 adimdan kalite beklenmez; boru hattini sinar)
import torch
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained('models/pilot-q35-4b', max_seq_length=1792, load_in_4bit=True)
tok = getattr(tok, 'tokenizer', tok)
FastLanguageModel.for_inference(model)
for soru in ('3/4+1/2', '2/9 ile 5/6 sayılarını topla.'):
    metin = tok.apply_chat_template([{'role': 'user', 'content': soru}], tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    girdi = tok(metin, return_tensors='pt', add_special_tokens=False).to('cuda')
    cikti = model.generate(**girdi, max_new_tokens=300, do_sample=False)
    print('SORU:', soru, '\n', tok.decode(cikti[0][girdi['input_ids'].shape[1]:], skip_special_tokens=True), '\n' + '-' * 60)
del model; torch.cuda.empty_cache()

In [ ]:
# 6) TAM EGITIM: arka planda baslar, bu hucre 5 dk'da bir ilerlemeyi yazar ve egitim bitene kadar calisir.
# Kopma olursa: hucre 1, 2, 3'u calistir, sonra DEVAM = True yapip bu hucreyi calistir (son checkpoint'ten surer).
import subprocess, time, json
DEVAM = False
ad = 'v2-q35-4b'
log, gecmis = f'reports/{ad}.log', f'reports/gecmis-{ad}.jsonl'
komut = f"python -X utf8 training/train.py --config configs/{ad}.json --ad {ad}" + (" --resume" if DEVAM else "")
p = subprocess.Popen(f"{komut} >> {log} 2>&1", shell=True)
print(time.strftime('%H:%M'), 'basladi | pid', p.pid, '|', komut, flush=True)
t0 = time.time()
while p.poll() is None:
    time.sleep(300)
    try:
        satirlar = [json.loads(x) for x in open(gecmis, encoding='utf-8') if x.strip()]
        tr = [x for x in satirlar if x.get('loss') is not None]
        ev = [x for x in satirlar if x.get('eval_loss') is not None]
        if tr:
            print(f"{time.strftime('%H:%M')} adim {tr[-1]['adim']}/2250  loss {tr[-1]['loss']:.4f}"
                  + (f"  | son eval {ev[-1]['eval_loss']:.4f} (adim {ev[-1]['adim']})" if ev else '')
                  + f"  | {(time.time()-t0)/60:.0f} dk", flush=True)
        else:
            print(time.strftime('%H:%M'), 'hazirlaniyor (model/veri/derleme)...', flush=True)
    except Exception as e:
        print('izleme:', e, flush=True)
print(time.strftime('%H:%M'), 'BITTI, cikis kodu', p.returncode)
!grep -E "^\[(bitti|vram|kayit|en iyi)\]|HATA|Error" {log} | tail -15